# Лекция 03. Алгоритмы и сложность

Учимся считать действия в программе, учитывать время и память и сравнивать решения до запуска на больших данных.

## Цели

После лекции вы сможете:

- определить размер входа и интересующий ресурс;
- посчитать действия в последовательных и вложенных циклах;
- различать время работы, память для результата и вспомогательную память;
- читать оценки `O`, `Ω` и `Θ` и различать лучший и худший случаи;
- объяснить стоимость среза, сортировки и проверки `in` для разных контейнеров;
- реализовать линейный и бинарный поиск и назвать их предусловия;
- учитывать подготовку данных при сравнении решений;
- сопоставить оценку сложности с небольшим вычислительным экспериментом.

После итогов находится необязательное продолжение о `lower_bound` и `bisect`.

## Перед началом

Понадобятся списки, индексы, циклы, условия, функции, множества и словари из первых двух занятий. В примерах работаем с обычными целыми числами: доступ по индексу, сравнение, арифметические операции и вычисление хеша считаем операциями постоянной стоимости. Стоимость очень больших целых и длинных строк здесь отдельно не изучаем.

Анализ сложности не заменяет измерения, а измерения не заменяют анализ: сегодня научимся использовать оба инструмента вместе.

Начнём с задачи, в которой короткий код может выполнять огромную работу. Нам понадобится перебор перестановок — всех возможных порядков элементов. Используем готовую функцию `itertools.permutations`. Она выдаёт кортежи по одному. Элементы на разных позициях считаются разными, поэтому при повторяющихся значениях некоторые кортежи будут совпадать.

In [ ]:
import itertools

for variant in itertools.permutations([1, 2, 3, 2]):
    print(variant)

## Сравним два списка

Нужно проверить, совпадают ли элементы двух списков **без учёта порядка, но с учётом числа повторов**. Например, `[1, 1, 2]` и `[2, 1, 1]` подходят, а `[1, 1, 2]` и `[1, 2, 2]` — нет.

Ниже — функция, которая перебирает перестановки второго списка и сравнивает каждую с первым. Сначала возьмём 4 элемента, затем 14. Пока наблюдаем, как меняется работа программы. К объяснению и более быстрому решению вернёмся в конце лекции, перед итогами.

In [ ]:
from itertools import permutations
from time import perf_counter


def same_elements_by_permutations(
    first: list[int], second: list[int]
) -> tuple[bool, int]:
    if len(first) != len(second):
        return False, 0

    target = tuple(first)
    checked = 0
    for candidate in permutations(second):
        checked += 1
        if candidate == target:
            return True, checked
    return False, checked


size = 4
second = list(range(size))
first = second[::-1]

started = perf_counter()
equal, checked = same_elements_by_permutations(first, second)
elapsed = perf_counter() - started

print("Первый список:", first)
print("Второй список:", second)
print(f"Совпадают без учёта порядка: {equal}")
print(f"Проверено перестановок: {checked:,}")
print(f"Время: {elapsed:.6f} с")

### Долгий опыт в фоне

Для 14 элементов запустим тот же перебор в **отдельном процессе Python**. Ячейка сразу закончит работу, а вычисления продолжатся параллельно остальным ячейкам. В конце лекции вернёмся к состоянию опыта и объясним его длительность.

Подготовленный [вспомогательный файл](examples/permutation_background.py) содержит управление процессом. Для работы с опытом достаточно трёх команд:

- `start_comparison(14)` — запустить; повторный вызов не создаёт второй процесс, пока первый работает;
- `show_comparison()` — показать состояние, прошедшее время или готовый результат;
- `stop_comparison()` — остановить вычисление.

Процесс может использовать другое ядро процессора и всё равно создаёт нагрузку на компьютер. В конце основного материала мы его остановим, если он ещё работает. При закрытии или перезапуске ядра Jupyter процесс также завершится.

Держите каталог `examples` рядом с ноутбуком. Следующая ячейка находит его при запуске из корня репозитория или из каталога `lesson03`.

In [ ]:
import sys
from pathlib import Path

examples_dir = Path("lesson03/examples")
if not (examples_dir / "permutation_background.py").is_file():
    examples_dir = Path("examples")
if not (examples_dir / "permutation_background.py").is_file():
    raise FileNotFoundError("Откройте ноутбук вместе с каталогом lesson03/examples.")

examples_path = str(examples_dir.resolve())
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

from permutation_background import start_comparison, show_comparison, stop_comparison

In [ ]:
start_comparison(14)
show_comparison()

## Зачем оценивать алгоритм

В начальном примере изменилось только число элементов: короткий перебор уже завершился, а для большого запущен отдельный процесс. Чтобы понять разницу в затратах, нужно посчитать работу, которую выполняет выбранный способ решения.

Два решения могут возвращать одинаковый ответ, но по-разному вести себя при росте данных. На списке из десяти элементов разница незаметна; на миллионах элементов один алгоритм может закончить работу, а другой — нет.

Нас интересует **как растут затраты вместе с размером входа**. Сначала научимся считать существенные действия, затем запишем их рост с помощью асимптотических оценок.

## Размер входа и базовая операция

Сначала выбирают параметр размера. Для списка это обычно его длина: `n = len(values)`. Затем определяют операцию, число выполнений которой отражает основную работу: например, сравнение элемента с целью.

В задачах с несколькими входами параметров может быть несколько: `n` строк и `m` столбцов, список из `n` элементов и `q` запросов. У каждого обозначения должен быть явный смысл.

В следующем примере считаем сравнения при поиске числа. Число действий зависит и от длины списка, и от того, где встретилась цель.

In [ ]:
def count_linear_search_comparisons(
    values: list[int], target: int
) -> int:
    comparisons = 0
    for value in values:
        comparisons += 1
        if value == target:
            return comparisons
    return comparisons


values = [4, 8, 15, 16, 23, 42]
print(count_linear_search_comparisons(values, 4))
print(count_linear_search_comparisons(values, 42))
print(count_linear_search_comparisons(values, 100))

## Сколько действий в циклах

Теперь посчитаем выполнения `count += 1` в нескольких фрагментах кода. Возьмём `n = 4`: одна цветная клетка на схеме соответствует одному выполнению этой операции.

![Один цикл выполняет действие n раз, два цикла подряд — 2n раз, два полных вложенных цикла — n² раз. При n = 4 получается 4, 8 и 16 действий.](illustrations/loop-work.png)

Два полных прохода подряд дают `n + n` действий. Если на каждом из `n` шагов внешнего цикла внутренний цикл делает ещё `n` шагов, получается `n × n`. Проверим эти числа исполняемым кодом.

In [ ]:
def count_one_pass(n: int) -> int:
    count = 0
    for i in range(n):
        count += 1
    return count


def count_two_passes(n: int) -> int:
    count = 0
    for i in range(n):
        count += 1
    for j in range(n):
        count += 1
    return count


def count_full_nested(n: int) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            count += 1
    return count


n = 4
print("Один проход:", count_one_pass(n))
print("Два прохода подряд:", count_two_passes(n))
print("Полные вложенные циклы:", count_full_nested(n))
assert count_one_pass(n) == n
assert count_two_passes(n) == 2 * n
assert count_full_nested(n) == n * n

### Внутренний цикл становится короче

Вложенные циклы не всегда выполняют ровно `n²` действий. Во фрагменте ниже внутренний цикл проходит только по индексам, которые больше `i`. При `n = 4` он делает 3 шага, затем 2, затем 1 и наконец 0.

In [ ]:
def count_shortening_nested(n: int) -> int:
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            count += 1
    return count


print(count_shortening_nested(4))
assert count_shortening_nested(4) == 6
assert count_shortening_nested(0) == 0
assert count_shortening_nested(1) == 0

![При n = 4 внутренний цикл посещает 3, 2, 1 и 0 клеток. Всего 6 действий; в общем случае n(n−1)/2.](illustrations/loop-triangle.png)

Для произвольного `n` число действий равно `(n - 1) + (n - 2) + ... + 1 = n(n - 1) / 2`.

| Фрагмент | Число действий | При `n = 4` | При `n = 8` |
| --- | --- | ---: | ---: |
| Один проход | `n` | 4 | 8 |
| Два прохода подряд | `2n` | 8 | 16 |
| Полные вложенные циклы | `n²` | 16 | 64 |
| Сокращающийся внутренний цикл | `n(n - 1) / 2` | 6 | 28 |

Мы считаем выбранную операцию, а не все инструкции интерпретатора. Управление циклами тоже требует работы, но в этих примерах не меняет порядок роста. Дальше научимся выражать этот порядок, не привязываясь к каждой константе.

## Время и память

Время оцениваем через число существенных операций. Для памяти различаем входные данные, память для результата и **вспомогательную память** сверх них.

Например, при суммировании чисел достаточно хранить один накопитель. При построении нового списка квадратов размер результата растёт вместе с входом. Если перед обходом ещё и скопировать исходный список, эта копия потребует отдельной вспомогательной памяти.

Поэтому одинаковое число проходов ещё не означает одинаковый расход памяти. После знакомства с оценками роста разберём, какие операции Python создают дополнительные объекты.

## Асимптотическая нотация

Для циклов мы получили точные числа действий: `n`, `2n`, `n²` и `n(n - 1) / 2`. Асимптотическая нотация позволяет сравнить их рост при больших входах.

Пусть $n$ — размер входа, $T(n) \ge 0$ — число операций, а $F(n) \ge 0$ — функция, с которой сравниваем рост. В этих формулах $N$ — фиксированный порог размера входа. На всех трёх графиках показана одна и та же функция $T(n)$; меняются только границы. Порог $N$ не обязан быть наименьшим возможным.

### Верхняя граница: $O$

$$
T(n) = O(F(n))
\iff
\exists C > 0\; \exists N \in \mathbb{N}:\;
\forall n > N\quad T(n) \le C \cdot F(n).
$$

![Верхняя граница O: при n > N синяя кривая T(n) не выше зелёной границы C F(n).](illustrations/asymptotic-o.png)

Существуют положительная константа $C$ и порог $N$, после которого для каждого размера входа число операций не превышает $C \cdot F(n)$.

### Нижняя граница: $\Omega$

$$
T(n) = \Omega(F(n))
\iff
\exists C > 0\; \exists N \in \mathbb{N}:\;
\forall n > N\quad T(n) \ge C \cdot F(n).
$$

![Нижняя граница Ω: при n > N синяя кривая T(n) не ниже оранжевой границы C F(n).](illustrations/asymptotic-omega.png)

Существуют положительная константа $C$ и порог $N$, после которого для каждого размера входа число операций не меньше $C \cdot F(n)$.

### Точный порядок роста: $\Theta$

$$
T(n) = \Theta(F(n))
\iff
\exists C_1 > 0\; \exists C_2 > 0\; \exists N \in \mathbb{N}:\;
\forall n > N\quad
C_1 \cdot F(n) \le T(n) \le C_2 \cdot F(n).
$$

![Точный порядок Θ: при n > N синяя кривая T(n) находится между C₁ F(n) и C₂ F(n).](illustrations/asymptotic-theta.png)

Одновременно выполняются верхняя и нижняя границы одного порядка.

Константы $C$, $C_1$, $C_2$ и порог $N$ выбираются один раз и не зависят от $n$. Неравенства должны выполняться **для всех** размеров входа $n > N$.

В прикладных обсуждениях словом «сложность» часто называют наиболее полезную тесную оценку $\Theta$, хотя записывают её как $O$. В строгом рассуждении эти обозначения различаются.

## Типичные порядки роста

| Порядок | Название | Пример | При увеличении `n` вдвое |
| --- | --- | --- | --- |
| `Θ(1)` | постоянный | доступ `values[index]` | почти без изменения |
| `Θ(log n)` | логарифмический | бинарный поиск | примерно +1 шаг |
| `Θ(n)` | линейный | полный проход | примерно ×2 |
| `Θ(n log n)` | линейно-логарифмический | эффективная сортировка | немного больше ×2 |
| `Θ(n²)` | квадратичный | все пары элементов | примерно ×4 |
| `Θ(2ⁿ)` | экспоненциальный | полный перебор подмножеств | значение `2ⁿ` возводится в квадрат |

![При удвоении n с 8 до 16 и 32 значения log₂ n равны 3, 4, 5; n — 8, 16, 32; n² — 64, 256, 1024.](illustrations/growth-doubling.png)

Для поиска и сортировки в таблице рассматривается худший случай. На схеме показаны значения трёх функций роста. Это удобная модель числа операций, а не измеренное время конкретной программы.

Основание логарифма в асимптотике обычно не указывают: логарифмы с разными постоянными основаниями отличаются постоянным множителем.

In [ ]:
from math import log2

for size in [8, 16, 32, 64]:
    print(
        f"n={size:>2}",
        f"log2(n)={log2(size):>3.0f}",
        f"n²={size ** 2:>4}",
    )

## Константы и младшие слагаемые

При асимптотическом анализе сохраняют доминирующий порядок роста:

- `3n + 20 = Θ(n)`;
- `n² + 100n = Θ(n²)`;
- `log₂n + 7 = Θ(log n)`.

Это не означает, что константы не важны на практике. Алгоритм с `100n` операций может проиграть алгоритму с `n²` на небольших входах. Асимптотика отвечает на вопрос о поведении при росте `n`, а не объявляет победителя для любого размера.

## Лучший, худший и средний случаи

Для одного размера входа время может зависеть от расположения данных. У линейного поиска:

- лучший случай — цель стоит первой: `Θ(1)`;
- худший случай — цель последняя или отсутствует: `Θ(n)`;
- средний случай требует модели вероятностей: какие входы и цели считаются вероятными.

Если модель не задана, безопаснее явно анализировать худший случай. Фраза «в среднем быстро» без описания распределения входов ничего не доказывает.

Выбор случая и выбор обозначения — разные вещи. `O` задаёт верхнюю границу для рассматриваемой функции времени, а `Ω` — нижнюю; сами обозначения не означают «худший» и «лучший» случай.

## Линейный поиск

Линейный поиск просматривает элементы слева направо и останавливается при первом совпадении. Он не требует сортировки и работает с любым перебираемым набором данных.

Если поиск ещё не завершился, после проверки первых `k` элементов мы знаем: среди них цели нет. Это простое утверждение является **инвариантом** цикла и помогает обосновать корректность.

In [ ]:
def linear_search(values: list[int], target: int) -> int:
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


print(linear_search([7, 2, 7, 9], 7))
print(linear_search([7, 2, 7, 9], 5))

## Сложность линейного поиска

В лучшем случае выполняется одно сравнение: `Θ(1)`. В худшем — `n` сравнений: `Θ(n)`. Дополнительная память равна `Θ(1)`: функция хранит только текущий индекс и элемент.

Ранний `return` улучшает отдельные запуски, но не меняет оценку худшего случая. Наличие короткого пути не позволяет назвать весь алгоритм константным.

## Бинарный поиск: цена ускорения

Если список **отсортирован**, проверка среднего элемента позволяет сузить область поиска:

- если `values[middle] < target`, исключаем середину и все элементы слева от неё;
- если `values[middle] > target`, исключаем середину и все элементы справа от неё;
- если `values[middle] == target`, возвращаем найденный индекс.

![В отсортированном списке при поиске 16 сравниваем цель с серединой 12. Все элементы до середины включительно не больше 12, поэтому исключаем их и продолжаем среди 16, 23 и 38.](illustrations/binary-search-idea.png)

Упорядоченность позволяет исключить сразу примерно половину оставшихся элементов. На неотсортированном списке такое рассуждение не работает: бинарный поиск может вернуть правдоподобный, но неверный результат.

## Полуинтервал поиска `[left, right)`

Будем хранить ещё не исключённые позиции в полуинтервале `[left, right)`: левая граница включена, правая не включена. В начале `left = 0`, `right = len(values)`. Пока `left < right`, интервал непуст.

Средний индекс `middle = (left + right) // 2`. На каждом шаге одна из границ сдвигается так, чтобы `middle` больше не оставался в следующем интервале. Это гарантирует завершение.

In [ ]:
def binary_search(values: list[int], target: int) -> int:
    left = 0
    right = len(values)

    while left < right:
        middle = (left + right) // 2
        if values[middle] == target:
            return middle
        if values[middle] < target:
            left = middle + 1
        else:
            right = middle

    return -1


print(binary_search([2, 5, 8, 12, 16, 23, 38], 16))
print(binary_search([2, 5, 8, 12, 16, 23, 38], 10))

## Трассировка бинарного поиска

Для цели `16` в списке `[2, 5, 8, 12, 16, 23, 38]` интервалы меняются так:

| `left` | `right` | `middle` | значение | действие |
| ---: | ---: | ---: | ---: | --- |
| 0 | 7 | 3 | 12 | отбросить позиции `0..3` |
| 4 | 7 | 5 | 23 | отбросить позиции `5..6` |
| 4 | 5 | 4 | 16 | цель найдена |

![Три шага поиска числа 16: рабочие диапазоны сужаются от [0, 7) до [4, 7) и [4, 5); ответ — индекс 4. Серая часть уже исключена, правая граница не входит в диапазон.](illustrations/binary-search-steps.png)

Серым отмечены ранее исключённые позиции. Граница `right` находится за последней ещё рассматриваемой позицией; обращаться к `values[right]` не требуется.

Таблица границ полезнее угадывания кода: она быстро обнаруживает зацикливание и ошибки на единицу.

## Почему получается `Θ(log n)`

После `k` шагов от `n` кандидатов остаётся примерно `n / 2ᵏ`. Поиск заканчивается, когда остаётся не больше одного кандидата:

`n / 2ᵏ <= 1`, следовательно, `k >= log₂n`.

Поэтому бинарный поиск выполняет `Θ(log n)` сравнений в худшем случае и использует `Θ(1)` дополнительной памяти в итеративной реализации. Для миллиарда элементов достаточно примерно 30 делений диапазона пополам.

In [ ]:
size = 1
for steps in range(0, 31, 5):
    if steps > 0:
        size = 2 ** steps
    print(f"{steps:>2} шагов -> до {size:,} элементов")

## Как выбрать поиск

| Ситуация | Подход | Время запроса в худшем случае |
| --- | --- | --- |
| Данные не отсортированы, один запрос | Линейный поиск | `O(n)` |
| Данные уже отсортированы | Бинарный поиск | `O(log n)` |
| Данные не отсортированы, нужен индекс в исходном списке | Линейный поиск | `O(n)` |
| Много запросов к неизменным данным | Один раз подготовить отсортированную копию | `O(log n)` после подготовки |

Бинарный поиск не делает сортировку бесплатной. Если ради одного запроса сначала сортировать данные, общая верхняя оценка времени — `O(n log n)` в худшем случае. Индексы в копии относятся к новому порядку элементов. Алгоритмы сортировки разберём позже.

Если нужен только ответ «есть ли такое значение», можно выбрать и другой контейнер. Посмотрим, сколько работы скрывается за привычными операциями Python.

## Сколько стоят операции Python

Количество строк кода не показывает объём работы. Вызов функции или оператор могут скрывать проход по данным, поиск или копирование.

В таблице `n` — длина контейнера, `k` — длина среза. Продолжаем работать с целыми числами, для которых сравнение и вычисление хеша считаем операциями постоянной стоимости.

| Операция | Время | Что происходит |
| --- | --- | --- |
| `len(values)` для списка, множества или словаря | `Θ(1)` | Читается уже известный размер |
| `values[index]` для списка | `Θ(1)` | Обращение к позиции |
| `sum(values)` | `Θ(n)` | Обход всех чисел |
| `values[left:right]` для списка | `Θ(k)` | Создание нового списка из `k` элементов |
| `sorted(values)` | `O(n log n)` в худшем случае | Создание и сортировка нового списка |
| `target in values` для списка | `O(n)` в худшем случае | Последовательная проверка элементов |
| `target in unique_values` для множества | `O(1)` в среднем, `O(n)` в худшем случае | Поиск по хешу |
| `key in mapping` для словаря | `O(1)` в среднем, `O(n)` в худшем случае | Поиск ключа по хешу |
| `value in mapping.values()` | `O(n)` в худшем случае | Последовательная проверка значений |

Для списка раннее совпадение сокращает поиск, а отсутствующее значение заставляет проверить все элементы. Множество и словарь используют хеш-таблицы. Хеш — вычисленное по значению число, которое помогает выбрать область поиска. Средняя оценка предполагает, что хеши распределяют элементы достаточно равномерно; в худшем случае могут потребоваться линейные затраты. Устройство хеш-таблиц разберём в занятии 7.

Оператор `in` у словаря проверяет **ключи**. Метод `.values()` даёт доступ к значениям, но отдельного быстрого индекса по ним у словаря нет.

In [ ]:
values = [10, 20, 30, 20]
unique_values = set(values)
labels = {10: "десять", 20: "двадцать", 30: "тридцать"}

print(20 in values)              # True: элемент списка.
print(20 in unique_values)       # True: элемент множества.
print(20 in labels)              # True: ключ словаря.
print("двадцать" in labels)      # False: такого ключа нет.
print("двадцать" in labels.values())  # True: такое значение есть.

assert len(values) == 4
assert len(unique_values) == 3
assert 20 in values and 20 in unique_values and 20 in labels
assert "двадцать" not in labels
assert "двадцать" in labels.values()

### Когда окупается создание множества

Проверка принадлежности готовому множеству обычно быстрая. Но само `set(values)` сначала проходит по всему списку: создание требует `O(n)` времени в среднем и до `O(n)` дополнительной памяти.

Для одного запроса преобразование списка в множество не даёт выигрыша в порядке роста: нужно всё равно обработать `n` элементов. Для `q` запросов к одним данным выбор уже существеннее:

- повторный поиск по списку требует до `O(nq)` времени;
- однократное создание множества и все запросы к нему требуют `O(n + q)` времени в среднем, ценой до `O(n)` дополнительной памяти.

В примере считаем, сколько запросов встретились в данных. Повторяющиеся запросы учитываются отдельно; число повторов значения внутри самих данных на ответ не влияет. Поэтому здесь множество сохраняет всё, что нужно для задачи.

In [ ]:
def count_requests_in_list(values: list[int], requests: list[int]) -> int:
    count = 0
    for target in requests:
        if target in values:
            count += 1
    return count


def count_requests_in_set(values: list[int], requests: list[int]) -> int:
    unique_values = set(values)
    count = 0
    for target in requests:
        if target in unique_values:
            count += 1
    return count


values = [10, 20, 30, 20]
requests = [20, 99, 20, 10]
assert count_requests_in_list(values, requests) == 3
assert count_requests_in_set(values, requests) == 3
print(count_requests_in_set(values, requests))

Вызов `set(values)` стоит **перед** циклом по запросам. Если написать `if target in set(values)` внутри цикла, множество будет создаваться заново для каждого запроса и затраты на подготовку снова вырастут до `O(nq)` в среднем. Значит, учитывать нужно и стоимость операции, и место её выполнения.

## Скрытая стоимость памяти

При создании множества мы обменяли дополнительную память на ускорение повторных запросов. Другие операции тоже могут создавать временные объекты. Выражение `values[1:]` создаёт новый список длины `n - 1`, поэтому цикл `for value in values[1:]:` использует `Θ(n)` дополнительной памяти. Обход по индексам или прямой обход исходного списка копии не создаёт.

Аналогично `sorted(values)` создаёт новый список. Иногда это правильная цена подготовки данных; важно назвать её явно.

![Входной список, список результата и временные данные учитываются отдельно. Список квадратов требует Θ(n) памяти для результата и Θ(1) сверх него; временный срез values[1:] добавляет Θ(n) вспомогательной памяти.](illustrations/memory-accounting.png)

Если функция строит и возвращает новый список из `n` квадратов, для результата нужна `Θ(n)` памяти. При обычном проходе остальные переменные занимают `Θ(1)`. Если же новый список создаётся только для промежуточного обхода, как в `for value in values[1:]:`, он относится к вспомогательной памяти.

| Вычисление | Время | Память для результата | Вспомогательная память |
| --- | --- | --- | --- |
| `sum(values)` | `Θ(n)` | `Θ(1)` | `Θ(1)` |
| `list(values)` — новая копия списка | `Θ(n)` | `Θ(n)` | `Θ(1)` сверх результата |
| `sum(values[1:])` | `Θ(n)` | `Θ(1)` | `Θ(n)` для временного среза |

Во всех трёх случаях время линейное. Расход памяти зависит от того, какие объекты создаются и какие из них являются результатом. Размер отдельных целых чисел здесь не учитываем.

In [ ]:
values = [10, 20, 30, 40]
tail = values[1:]
tail[0] = 999
print(values)
print(tail)

In [ ]:
values = [40, 10, 30, 20]
ordered = sorted(values)  # Создан новый список.

print(values)
print(ordered)
print(ordered is values)

## Измерение времени

Для небольшого эксперимента используют `time.perf_counter()`, повторяют операцию много раз и сравнивают входы разных размеров. Важно:

- измерять одну и ту же задачу;
- отделять подготовку данных от запроса или явно включать её в обе стратегии;
- повторять измерения, потому что система шумит;
- не делать общий вывод по одному маленькому входу;
- сначала проверить корректность сравниваемых функций.

Время в секундах зависит от машины, а характер роста должен быть устойчивее.

Ниже измеряем повторные запросы к **уже подготовленному отсортированному списку**. Цель отсутствует, поэтому линейный поиск проходит весь список. Сравниваем две собственные функции, разобранные выше. Этот замер показывает разницу на одном размере; для проверки характера роста нужно повторить эксперимент на нескольких размерах.

Если фоновый перебор ещё работает, он создаёт дополнительную нагрузку и может влиять на замеры. Для отдельного чистого эксперимента сначала выполните `stop_comparison()`. Во время лекции учитывайте эту нагрузку при обсуждении полученных секунд.

In [ ]:
from time import perf_counter

values = list(range(100_000))
targets = [-1] * 100
assert linear_search(values, -1) == binary_search(values, -1) == -1

started = perf_counter()
for target in targets:
    linear_search(values, target)
linear_seconds = perf_counter() - started

started = perf_counter()
for target in targets:
    binary_search(values, target)
binary_seconds = perf_counter() - started

print(f"linear: {linear_seconds:.6f} s")
print(f"binary: {binary_seconds:.6f} s")

## Неожиданно, но по правилам

Перед запуском предскажите результат.

1. Бинарный поиск не проверяет предусловие сортировки. На неупорядоченном списке он не обязан сообщать об ошибке и может просто не найти существующее значение.
2. Один цикл не гарантирует линейное время. Если на каждой итерации создавать срез оставшейся части, суммарно копируется `n + (n - 1) + ... + 1 = Θ(n²)` элементов.
3. Сортировка копии разрешает бинарный поиск, но найденный индекс относится к отсортированной копии, а не к исходному списку.

In [ ]:
unsorted_values = [1, 4, 2, 5, 7]
print(binary_search(unsorted_values, 4))  # 4 есть, но поиск вернул -1.

def count_copied_items(size: int) -> int:
    values = list(range(size))
    copied_items = 0
    for index in range(size):
        tail = values[index:]
        copied_items += len(tail)
    return copied_items

print(count_copied_items(5), count_copied_items(10))

target = 4
ordered = sorted(unsorted_values)
position = binary_search(ordered, target)
print(ordered, position, unsorted_values.index(target))

Мы разобрали подсчёт действий, стоимость операций и измерение времени. Теперь проверим, что произошло с долгим опытом, и вернёмся к его устройству. Если вы пропустили фоновый запуск, следующая ячейка просто сообщит об этом.

In [ ]:
show_comparison()

## Возвращаемся к сравнению списков

Теперь применим подсчёт операций, оценки роста и учёт памяти к задаче из начала лекции. Функция создаёт перестановки второго списка и сравнивает каждую с первым. Вместе с признаком совпадения функция из начала лекции возвращает число проверенных перестановок.

В наших примерах второй список идёт по возрастанию, а первый — в обратном порядке. Для такого второго списка обратный порядок будет **последней перестановкой**. Поэтому совпадение есть, но для его обнаружения приходится проверить все варианты.

Для `n` разных элементов существует `n!` перестановок: на первое место можно поставить любой из `n` элементов, на второе — любой из оставшихся `n − 1`, и так далее. При четырёх элементах получается `4! = 4 × 3 × 2 × 1 = 24` проверки.

| Размер списка | Число перестановок |
| ---: | ---: |
| 4 | 24 |
| 13 | 6 227 020 800 |
| 14 | 87 178 291 200 |

Даже если проверять **30 миллионов перестановок в секунду**, на `14!` вариантов уйдёт почти **50 минут**. Это расчёт для заданной скорости, а не обещанное время работы программы: реальная скорость зависит от компьютера и нагрузки. Всего один дополнительный элемент при переходе от 13 к 14 увеличивает число вариантов в 14 раз.

Здесь `n!` — число проверяемых перестановок. Сравнение одной пары последовательностей может потребовать до `n` сравнений элементов, поэтому верхняя оценка времени — `O(n · n!)`.

`itertools.permutations` выдаёт перестановки по одной в виде кортежей. Поэтому первый список один раз преобразуется в кортеж для сравнения, а все перестановки одновременно в памяти не хранятся. Дополнительная память — `O(n)`: текущая перестановка, первый список в виде кортежа и состояние перебора.

Мы объяснили, почему небольшое увеличение входа так сильно меняет время. Теперь нужно изменить сам способ сравнения. Как получить для двух списков один и тот же порядок элементов, сохранив все повторения?

### Более быстрое решение

Отсортируем оба списка и сравним полученные копии. Если значения и число их повторов совпадают, после сортировки списки будут одинаковыми. Если отличаются, сравнение это обнаружит.

Простого сравнения множеств здесь недостаточно: у `[1, 1, 2]` и `[1, 2, 2]` одинаковое множество значений, но исходное условие требует считать повторы. В задаче с запросами множество подходило, а здесь потеряло бы нужную информацию.

In [ ]:
def same_elements_by_sorting(first: list[int], second: list[int]) -> bool:
    if len(first) != len(second):
        return False
    return sorted(first) == sorted(second)


assert same_elements_by_sorting([], [])
assert same_elements_by_sorting([1, 1, 2], [2, 1, 1])
assert not same_elements_by_sorting([1, 1, 2], [1, 2, 2])
assert not same_elements_by_sorting([1], [1, 1])
assert same_elements_by_sorting([-2, 0, -2], [-2, -2, 0])

first = list(range(14))
second = first[::-1]
started = perf_counter()
equal = same_elements_by_sorting(first, second)
elapsed = perf_counter() - started
print(f"Совпадают без учёта порядка: {equal}")
print(f"Время: {elapsed:.6f} с")
assert first == list(range(14))
assert second == list(range(13, -1, -1))

Для двух списков одинаковой длины `n` сортировки требуют `O(n log n)` времени в худшем случае, сравнение — до `O(n)`. Общая верхняя оценка остаётся `O(n log n)`. Две отсортированные копии занимают `O(n)` дополнительной памяти; исходные списки сохраняются. При разных длинах функция заканчивает работу сразу.

Python умеет использовать уже имеющийся порядок при сортировке, поэтому возрастающий и убывающий списки из примера обрабатываются особенно быстро. Один такой замер не доказывает оценку худшего случая. Но мы уже можем объяснить главное: новая функция не перебирает перестановки и при этом решает ту же задачу, включая проверку повторов.

Объём работы теперь понятен; дожидаться всех перестановок для этого не нужно. Завершим фоновый процесс, если он ещё работает. Если вычисление уже закончено, команда сохранит и покажет его результат.

In [ ]:
stop_comparison()

## Итоги

- Сложность описывает рост затрат вместе с размером входа.
- Для циклов нужно считать работу: последовательные проходы складываются, а вложенные зависят от границ каждого прохода.
- Время, память для результата и вспомогательную память оценивают отдельно.
- `O`, `Ω` и `Θ` задают верхнюю, нижнюю и тесную асимптотические границы; лучший и худший случаи анализируют отдельно.
- Стоимость `in` зависит от контейнера: список перебирает элементы, множества и словари используют хеш-таблицы. У словаря проверяются ключи.
- Линейный поиск не требует порядка и работает за `Θ(n)` в худшем случае. Бинарный поиск требует сортировки и работает за `Θ(log n)` в худшем случае.
- Подготовка данных и временные копии имеют собственную стоимость.
- Ускорение должно сохранять условие задачи: при сравнении списков нельзя потерять число повторов.
- Эксперимент дополняет анализ, если корректность и условия измерения контролируются.

На [семинаре](seminar.ipynb) повторим условия и циклы, оценим время и память небольших функций и попробуем ускорить решения задач.

Основной материал лекции на этом закончен. Ниже — справочное продолжение о границах бинарного поиска для тех, кому интересно.

## Дополнительно: `lower_bound` и границы поиска

Этот раздел можно прочитать после лекции. Для основных примеров выше достаточно обычного бинарного поиска, который возвращает любой найденный индекс или `-1`.

Иногда нужна более точная граница: первое вхождение повторяющегося значения, позиция для вставки или число одинаковых элементов. Для этого немного изменим уже знакомый поиск.

### Повторяющиеся значения

Простой бинарный поиск возвращает **какое-нибудь** совпадение. Если нужен первый индекс, при равенстве нельзя сразу завершаться: найденная позиция становится кандидатом, а поиск продолжается слева.

Более общий подход — искать первую позицию, на которой значение не меньше цели. Такая граница называется `lower bound`. После поиска остаётся проверить, действительно ли на найденной позиции стоит цель.

### Что возвращает граница

`lower_bound(values, target)` возвращает первый индекс, где значение не меньше `target`. Если такого элемента нет, результат равен `len(values)`. Это допустимая граница, но не индекс существующего элемента.

Например, для `[1, 2, 2, 2, 5]` цель `2` даёт границу `1`, цель `3` — границу `4`, а цель `6` — границу `5`. Чтобы проверить наличие самой цели, сначала нужно убедиться, что граница находится внутри списка, и только затем сравнить элемент.

In [ ]:
def lower_bound(values: list[int], target: int) -> int:
    left = 0
    right = len(values)

    while left < right:
        middle = (left + right) // 2
        if values[middle] < target:
            left = middle + 1
        else:
            right = middle

    return left


values = [1, 2, 2, 2, 5]
for target in [0, 2, 3, 6]:
    print(target, lower_bound(values, target))

In [ ]:
values = [1, 2, 2, 2, 5]
for target in [0, 2, 3, 6]:
    position = lower_bound(values, target)
    found = position < len(values) and values[position] == target
    print(target, position, found)

assert lower_bound([], 10) == 0
assert lower_bound(values, 2) == 1
assert lower_bound(values, 6) == len(values)

### Стандартная библиотека `bisect`

В рабочем коде границы отсортированного списка обычно ищут модулем `bisect`. `bisect_left(values, target)` возвращает первую позицию, куда можно вставить цель, сохранив порядок; `bisect_right` — позицию после всех равных элементов.

Собственную реализацию пишем, чтобы понять инвариант и границы. После этого стандартная функция уменьшает риск ошибки и яснее выражает намерение.

In [ ]:
from bisect import bisect_left, bisect_right

values = [1, 2, 2, 2, 5]
left = bisect_left(values, 2)
right = bisect_right(values, 2)
print(left, right, right - left)

В примере `bisect_left` и `bisect_right` дают границы полуинтервала всех двоек: `[1, 4)`. Его длина равна `4 - 1 = 3`. Каждый поиск границы требует `O(log n)` времени в худшем случае.

Найти позицию вставки и вставить элемент — разные операции. Поиск позиции быстрый, но вставка в середину обычного списка может сдвинуть `O(n)` элементов.